Pre Processing Notebook for NOC Entities

In [ ]:
'''
The fundamental goal is to create a datatable that acts as a map that outlines the path 
from Broad Category to Unit Group as defined by StatsCan here: 
https://noc.esdc.gc.ca/Structure/Hierarchy
'''

In [7]:
import pandas as pd 
import polars as pl

In [8]:
dp_NOC_Raw = pl.read_csv('noc_2021_classification_structure.csv')

In [10]:
dp_NOC_Raw.schema

Schema([('Level', Int64),
        ('Hierarchical structure', String),
        ('Code - NOC 2021 V1.0', Int64),
        ('Class title', String),
        ('Class definition', String)])

In [14]:
#Verifying the number of Unique entries per Hierarchy 

df_Hierachy_Counts = dp_NOC_Raw.group_by(['Level','Hierarchical structure']).agg(
    pl.len().alias('Number of Classes')
).to_pandas().sort_values(by='Level', ascending=True).reset_index(drop=True)

df_Hierachy_Counts

,Level,Hierarchical structure,Number of Classes
0,1,Broad Category,10
1,2,Major Group,45
2,3,Sub-major Group,89
3,4,Minor Group,162
4,5,Unit Group,516


In [ ]:
#This is correct according to the source: https://noc.esdc.gc.ca/Structure/Hierarchy

In [18]:
print(dp_NOC_Raw.select(['Level','Hierarchical structure','Code - NOC 2021 V1.0','Class title','Class definition']).head(15))

shape: (15, 5)
┌───────┬─────────────────┬────────────────────────┬───────────────────────┬───────────────────────┐
│ Level ┆ Hierarchical    ┆ Code - NOC 2021 V1.0   ┆ Class title           ┆ Class definition      │
│ ---   ┆ structure       ┆ ---                    ┆ ---                   ┆ ---                   │
│ i64   ┆ ---             ┆ i64                    ┆ str                   ┆ str                   │
│       ┆ str             ┆                        ┆                       ┆                       │
╞═══════╪═════════════════╪════════════════════════╪═══════════════════════╪═══════════════════════╡
│ 1     ┆ Broad Category  ┆ 0                      ┆ Legislative and       ┆ This broad category   │
│       ┆                 ┆                        ┆ senior managem…       ┆ comprises …           │
│ 2     ┆ Major Group     ┆ 0                      ┆ Legislative and       ┆ This major group      │
│       ┆                 ┆                        ┆ senior manager…       ┆

In [ ]:
'''
The structure of the table does not clearly outline the hierarchies between each other, 
nor does it provide a clear pathway from Broad Category to Unit Group:
We want a table that has columns like this: 
Broad Category Code, Broad Category Class, Broad Category Class Definition, 
Major Group Code, Major Group Class, Major Group Class Definition, .... 
until the last 3 columnd str Unit Group Code, Unit Group Class and Unit Group Class Definition.
'''

Creating Individual Frames, 1 per Hierarchy 

In [19]:
#Storing columns name for ease of use 
code_col = "Code - NOC 2021 V1.0"
title_col = "Class title"
def_col = "Class definition"

In [46]:
#Level 1 - Broad Category 
level = 1
broad = (
    dp_NOC_Raw.filter(pl.col("Level") == level)
    .select([
        #Pads the code in keeping with the level character length, so we can perserve structures
        pl.col(code_col).cast(pl.Utf8).alias("Broad Category Code").str.pad_start(level, "0"),
        pl.col(title_col).alias("Broad Category Class"),
        pl.col(def_col).alias("Broad Category Definition")
    ])
)
broad.head(5)

Broad Category Code,Broad Category Class,Broad Category Definition
str,str,str
"""0""","""Legislative and senior managem…","""This broad category comprises …"
"""1""","""Business, finance and administ…","""This broad category comprises …"
"""2""","""Natural and applied sciences a…","""This broad category comprises …"
"""3""","""Health occupations""","""This broad category comprises …"
"""4""","""Occupations in education, law …","""This broad category comprises …"


In [51]:
#Level 2 - Major Group 
level = 2 
major = (
    dp_NOC_Raw.filter(pl.col("Level") == level)
    .select([
        #Pads the code in keeping with the level character length, so we can perserve structures
        pl.col(code_col).cast(pl.Utf8).alias("Major Group Code").str.pad_start(level, "0"),
        pl.col(title_col).alias("Major Group Class"),
        pl.col(def_col).alias("Major Group Definition")
    ])
    .with_columns(
        #Helps us extract the NOC Code for the broad category. str.slice(start postion, number of chars to store)
        pl.col("Major Group Code").str.slice(0,1).alias("Broad Category Code") 
    )
)
major.head(5)

Major Group Code,Major Group Class,Major Group Definition,Broad Category Code
str,str,str,str
"""00""","""Legislative and senior manager…","""This major group comprises sen…","""0"""
"""10""","""Specialized middle management …","""This major group comprises spe…","""1"""
"""11""","""Professional occupations in fi…","""This major group comprises pro…","""1"""
"""12""","""Administrative and financial s…","""This major group comprises adm…","""1"""
"""13""","""Administrative occupations and…","""This major group comprises adm…","""1"""


In [52]:
#Level 3 - Sub-Major Group
level = 3 
sub_major = (
    dp_NOC_Raw.filter(pl.col("Level") == level)
    .select([
        #Pads the code in keeping with the level character length, so we can perserve structures
        pl.col(code_col).cast(pl.Utf8).alias("Sub-major Group Code").str.pad_start(level, "0"),
        pl.col(title_col).alias("Sub-major Group Class"),
        pl.col(def_col).alias("Sub-major Group Definition")
    ])
    .with_columns(
        pl.col("Sub-major Group Code").str.slice(0,2).alias("Major Group Code")
    )
)
sub_major.head(5)

Sub-major Group Code,Sub-major Group Class,Sub-major Group Definition,Major Group Code
str,str,str,str
"""000""","""Legislative and senior manager…","""This sub-major group comprises…","""00"""
"""100""","""Specialized middle management …","""This sub-major group comprises…","""10"""
"""111""","""Professional occupations in fi…","""This sub-major group comprises…","""11"""
"""112""","""Professional occupations in bu…","""This sub-major group comprises…","""11"""
"""120""","""Administrative and financial s…","""This sub-major group comprises…","""12"""


In [53]:
#Level 4 - Minor Group
level = 4
minor = (
    dp_NOC_Raw.filter(pl.col("Level") == 4)
    .select([
        pl.col(code_col).cast(pl.Utf8).alias("Minor Group Code").str.pad_start(level, "0"),
        pl.col(title_col).alias("Minor Group Class"),
        pl.col(def_col).alias("Minor Group Definition")
    ])
    .with_columns(
        pl.col("Minor Group Code").str.slice(0,3).alias("Sub-major Group Code")
    )
)

minor.head(5)

Minor Group Code,Minor Group Class,Minor Group Definition,Sub-major Group Code
str,str,str,str
"""0001""","""Legislative and senior manager…","""This minor group comprises leg…","""000"""
"""1001""","""Administrative services manage…","""This minor group comprises adm…","""100"""
"""1002""","""Managers in financial and busi…","""This minor group comprises man…","""100"""
"""1003""","""Managers in communication (exc…","""This minor group comprises man…","""100"""
"""1110""","""Auditors, accountants and inve…","""This minor group comprises aud…","""111"""


In [54]:
#Level 5: Unit Group 
level = 5
unit = (
    dp_NOC_Raw.filter(pl.col("Level") == 5)
    .select([
        pl.col(code_col).cast(pl.Utf8).alias("Unit Group Code").str.pad_start(level, "0"),
        pl.col(title_col).alias("Unit Group Class"),
        pl.col(def_col).alias("Unit Group Definition")
    ])
    .with_columns(
        pl.col("Unit Group Code").str.slice(0,4).alias("Minor Group Code")
    )
)
unit.head(5)

Unit Group Code,Unit Group Class,Unit Group Definition,Minor Group Code
str,str,str,str
"""00010""","""Legislators""","""Legislators participate in the…","""0001"""
"""00011""","""Senior government managers and…","""Senior government managers and…","""0001"""
"""00012""","""Senior managers - financial, c…","""Senior managers in financial, …","""0001"""
"""00013""","""Senior managers - health, educ…","""Senior managers in health, edu…","""0001"""
"""00014""","""Senior managers - trade, broad…","""Senior managers in trade, broa…","""0001"""


In [58]:
df_Heiarchy_Pathways = (
    unit
    .join(minor, on="Minor Group Code", how="left")
    .join(sub_major, on="Sub-major Group Code", how="left")
    .join(major, on="Major Group Code", how="left")
    .join(broad, on="Broad Category Code", how="left")
).to_pandas()

In [61]:
df_Heiarchy_Pathways

,Unit Group Code,Unit Group Class,Unit Group Definition,Minor Group Code,Minor Group Class,Minor Group Definition,Sub-major Group Code,Sub-major Group Class,Sub-major Group Definition,Major Group Code,Major Group Class,Major Group Definition,Broad Category Code,Broad Category Class,Broad Category Definition
0,00010,Legislators,Legislators participate in the activities of a...,0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...
1,00011,Senior government managers and officials,"Senior government managers and officials plan,...",0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...
2,00012,"Senior managers - financial, communications an...","Senior managers in financial, communications a...",0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...
3,00013,"Senior managers - health, education, social an...","Senior managers in health, education, social a...",0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...
4,00014,"Senior managers - trade, broadcasting and othe...","Senior managers in trade, broadcasting and oth...",0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511,95104,Labourers in rubber and plastic products manuf...,Labourers in rubber and plastic products manuf...,9510,"Labourers in processing, manufacturing and uti...",This minor group comprises labourers in proces...,951,"Labourers in processing, manufacturing and uti...",This sub-major group comprises labourers in pr...,95,"Labourers in processing, manufacturing and uti...",This major group comprises labourers in proces...,9,Occupations in manufacturing and utilities,This broad category comprises middle managemen...
512,95105,Labourers in textile processing and cutting,Labourers in textile processing perform a vari...,9510,"Labourers in processing, manufacturing and uti...",This minor group comprises labourers in proces...,951,"Labourers in processing, manufacturing and uti...",This sub-major group comprises labourers in pr...,95,"Labourers in processing, manufacturing and uti...",This major group comprises labourers in proces...,9,Occupations in manufacturing and utilities,This broad category comprises middle managemen...
513,95106,Labourers in food and beverage processing,Labourers in food and beverage processing perf...,9510,"Labourers in processing, manufacturing and uti...",This minor group comprises labourers in proces...,951,"Labourers in processing, manufacturing and uti...",This sub-major group comprises labo

In [ ]:
'''
It looks like our approach worked, but we should check the following:
1. Whether the unique counts remained the same for each Teir/Hiearchy 
2. If everything is mapped correctly. 
'''

In [62]:
df_Heiarchy_Pathways.columns

Index(['Unit Group Code', 'Unit Group Class', 'Unit Group Definition',
       'Minor Group Code', 'Minor Group Class', 'Minor Group Definition',
       'Sub-major Group Code', 'Sub-major Group Class',
       'Sub-major Group Definition', 'Major Group Code', 'Major Group Class',
       'Major Group Definition', 'Broad Category Code', 'Broad Category Class',
       'Broad Category Definition'],
      dtype='object')

In [64]:
# Checking 1. 
Heiarchy_Columns = {
    'Broad Category Class': None,
    'Major Group Class': None,
    'Sub-major Group Class': None,
    'Minor Group Class': None,
    'Unit Group Class': None
}

for Heiarchy in Heiarchy_Columns.keys():
    Heiarchy_Columns[Heiarchy] = df_Heiarchy_Pathways[Heiarchy].nunique()

Heiarchy_Columns

{'Broad Category Class': 10,
 'Major Group Class': 45,
 'Sub-major Group Class': 89,
 'Minor Group Class': 162,
 'Unit Group Class': 516}

In [65]:
#Checking 2 
dp_Heiarchy_Pathways = pl.from_pandas(df_Heiarchy_Pathways)

#Checking to see if the code lenght is correct
dp_Heiarchy_Pathways = dp_Heiarchy_Pathways.with_columns(
    (
        (pl.col("Broad Category Code").str.len_chars() == 1) &
        (pl.col("Major Group Code").str.len_chars() == 2) &
        (pl.col("Sub-major Group Code").str.len_chars() == 3) &
        (pl.col("Minor Group Code").str.len_chars() == 4) &
        (pl.col("Unit Group Code").str.len_chars() == 5)
    ).alias("Length Valid")
)

#Checking to see if the code mapping is correct 
dp_Heiarchy_Pathways= dp_Heiarchy_Pathways.with_columns(
    (   
        #starts_with function checks to see if code starts with a substring, ex. 211 -> 21
        pl.col("Major Group Code").str.starts_with(pl.col("Broad Category Code")) &
        pl.col("Sub-major Group Code").str.starts_with(pl.col("Major Group Code")) &
        pl.col("Minor Group Code").str.starts_with(pl.col("Sub-major Group Code")) &
        pl.col("Unit Group Code").str.starts_with(pl.col("Minor Group Code")) &
        (pl.col("Broad Category Code").str.len_chars() == 1) &
        (pl.col("Major Group Code").str.len_chars() == 2) &
        (pl.col("Sub-major Group Code").str.len_chars() == 3) &
        (pl.col("Minor Group Code").str.len_chars() == 4) &
        (pl.col("Unit Group Code").str.len_chars() == 5)
    ).alias("Hierarchy Valid")
)

In [66]:
dp_Heiarchy_Pathways.filter((pl.col('Length Valid')==False) | (pl.col('Hierarchy Valid')==False))

Unit Group Code,Unit Group Class,Unit Group Definition,Minor Group Code,Minor Group Class,Minor Group Definition,Sub-major Group Code,Sub-major Group Class,Sub-major Group Definition,Major Group Code,Major Group Class,Major Group Definition,Broad Category Code,Broad Category Class,Broad Category Definition,Length Valid,Hierarchy Valid
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool


In [ ]:
#Looks like the mapping was done correctly 

In [67]:
df_Final_Heiarchy_Pathways = dp_Heiarchy_Pathways.to_pandas()

In [68]:
df_Final_Heiarchy_Pathways

,Unit Group Code,Unit Group Class,Unit Group Definition,Minor Group Code,Minor Group Class,Minor Group Definition,Sub-major Group Code,Sub-major Group Class,Sub-major Group Definition,Major Group Code,Major Group Class,Major Group Definition,Broad Category Code,Broad Category Class,Broad Category Definition,Length Valid,Hierarchy Valid
0,00010,Legislators,Legislators participate in the activities of a...,0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...,True,True
1,00011,Senior government managers and officials,"Senior government managers and officials plan,...",0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...,True,True
2,00012,"Senior managers - financial, communications an...","Senior managers in financial, communications a...",0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...,True,True
3,00013,"Senior managers - health, education, social an...","Senior managers in health, education, social a...",0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...,True,True
4,00014,"Senior managers - trade, broadcasting and othe...","Senior managers in trade, broadcasting and oth...",0001,Legislative and senior managers,This minor group comprises legislators and sen...,000,Legislative and senior managers,This sub-major group comprises legislators and...,00,Legislative and senior managers,This major group comprises senior management o...,0,Legislative and senior management occupations,This broad category comprises legislators and ...,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511,95104,Labourers in rubber and plastic products manuf...,Labourers in rubber and plastic products manuf...,9510,"Labourers in processing, manufacturing and uti...",This minor group comprises labourers in proces...,951,"Labourers in processing, manufacturing and uti...",This sub-major group comprises labourers in pr...,95,"Labourers in processing, manufacturing and uti...",This major group comprises labourers in proces...,9,Occupations in manufacturing and utilities,This broad category comprises middle managemen...,True,True
512,95105,Labourers in textile processing and cutting,Labourers in textile processing perform a vari...,9510,"Labourers in processing, manufacturing and uti...",This minor group comprises labourers in proces...,951,"Labourers in processing, manufacturing and uti...",This sub-major group comprises labourers in pr...,95,"Labourers in processing, manufacturing and uti...",This major group comprises labourers in proces...,9,Occupations in manufacturing and utilities,This broad category comprises middle managemen...,True,True
513,95106,Labourers in food and beverage processing,Labourers in food and beverage processing perf...,9510,"Labourers in processing, manufacturing and uti...",This minor group comprises laboure

In [69]:
df_Final_Heiarchy_Pathways.to_csv('Tbl_NOC_Heiarchy_Pathways.csv',index=False)